In [1]:
from langchain.vectorstores import Chroma
from langchain.embeddings import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.llms import OpenAI
from langchain.chains import VectorDBQA, RetrievalQA
from langchain.document_loaders import TextLoader

## Load documents

load documents into kernel

In [2]:
# For Python files
from langchain_community.document_loaders import PythonLoader
python_docs = PythonLoader('ol_label_images.py').load()

## Split documents

split documents into small chunks

In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(python_docs)

## Initialize embeddings model

define embeddings model to process each chunk 

In [4]:
# Load model directly
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

In [5]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from langchain.embeddings.base import Embeddings

class CustomHuggingFaceEmbeddings(Embeddings):
    def __init__(self, model_name="sentence-transformers/all-MiniLM-L6-v2", device="cpu"):
        # Load tokenizer and model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.device = device
        self.model.to(device)
        # Set to evaluation mode
        self.model.eval()
    
    def _get_embedding(self, text):
        # Tokenize input text
        inputs = self.tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        
        # Get embeddings
        with torch.no_grad():
            outputs = self.model(**inputs)
            # Mean pooling - take average of all token embeddings
            token_embeddings = outputs.last_hidden_state
            attention_mask = inputs['attention_mask']
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
            sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
            embeddings = sum_embeddings / sum_mask
            
        # Convert to numpy array and return
        return embeddings[0].cpu().numpy()
    
    def embed_documents(self, texts):
        """Generate embeddings for a list of documents."""
        return [self._get_embedding(text) for text in texts]
    
    def embed_query(self, text):
        """Generate embedding for a query."""
        return self._get_embedding(text)

# Create the embeddings object
embeddings = CustomHuggingFaceEmbeddings()

## Initialize ChromaDB

Insert chunks into the Chroma vector database.

In [6]:
# Now you can use it with Chroma
vectordb = Chroma.from_documents(texts, embeddings)

## Create the chain



In [18]:
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
# Import RetrievalQA instead of VectorDBQA
from langchain.chains import RetrievalQA

# a smaller model suitable for M3 Mac
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,  # Use float16 for M3
    device_map="auto"
)
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.25,  # Low temperature for more focused answers
    repetition_penalty=1.2
)
llm = HuggingFacePipeline(pipeline=pipe)

qa = RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="stuff", 
    retriever=vectordb.as_retriever()  # Use as_retriever() instead of passing vectorstore directly
)

Device set to use mps


## Question Asking! 

In [22]:
import warnings
warnings.filterwarnings("ignore", message="To copy construct from a tensor")

In [23]:
query = "What does detect_garament do? What purpose does it serve?"
ans = qa.invoke(query)

In [24]:
def clean_qa_output(qa_response):
    # If the response is a dictionary with a 'result' key
    if isinstance(qa_response, dict) and 'result' in qa_response:
        return qa_response['result'].strip()
    # If the response is a string
    elif isinstance(qa_response, str):
        return qa_response.strip()
    # If it's some other type of object with content
    elif hasattr(qa_response, 'content'):
        return qa_response.content.strip()
    else:
        return str(qa_response)

In [25]:
formatted_ans = clean_qa_output(ans)
print(formatted_ans)

Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

def first_stage_obj_det(self):
        """
        Detects objects in an image using the Google Vision API and retrieves bounding box information.

        Parameters:
        - image: The image to analyze, provided as a Google Vision `Image` object.
        - client: An instance of the Google Vision API client.

        Returns:
        - objects: A list of localized object annotations detected in the image.
        - result: A dictionary mapping object names to their bounding box vertices.

        """ 
        google_image = vision.Image(content=self.image)
        response = self.client.object_localization(image=google_image)

        if response.error.message:
            raise Exception(f"Google Vision API Error: {response.error.message}")

        objects = response.localized_object_annotations

        # Print only t